In [ ]:
import torch
import numpy as np
from pathlib import Path
from PIL import Image
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms.functional as TF
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchmetrics.detection.mean_ap import MeanAveragePrecision

# ── Config ────────────────────────────────────────────────────────────────────
DEVICE           = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT       = "pollen_best.pth"
TARGET_NUM_CLASSES = 53  # 52 classes + background
IMGSZ            = 340
BATCH_SIZE       = 8
SCORE_THRESHOLD  = 0.5   # ignore predictions below this confidence

POLLEN_VAL_TXT = "/storage/praha1/home/mamedove/00_Data/2025_Pollen/Multi_class/multi-class-set/val.txt"
TARGET_CLASSES = [str(i) for i in range(1, 53)]

# ── Load model ────────────────────────────────────────────────────────────────
def build_model(num_classes):
    model = fasterrcnn_resnet50_fpn(weights=None)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

model = build_model(TARGET_NUM_CLASSES)
model.load_state_dict(torch.load(CHECKPOINT, map_location=DEVICE))
model.to(DEVICE)
model.eval()
print("Model loaded.")

# ── Dataset ───────────────────────────────────────────────────────────────────
class PollenDatasetFromTxt(Dataset):
    def __init__(self, txt_path, class_names, imgsz=320):
        self.imgsz       = imgsz
        self.class_names = class_names
        txt_path         = Path(txt_path)
        base_dir         = txt_path.parent

        with open(txt_path) as f:
            all_paths = [base_dir / l.strip() for l in f if l.strip()]

        self.images = []
        for img_path in all_paths:
            img_path   = img_path.resolve()
            label_path = img_path.with_suffix(".txt")
            if img_path.exists() and label_path.exists():
                self.images.append(img_path)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path   = self.images[idx]
        label_path = img_path.with_suffix(".txt")

        img        = Image.open(img_path).convert("RGB")
        w, h       = img.size

        boxes, labels = [], []
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                cls, cx, cy, bw, bh = map(float, parts)
                x1 = (cx - bw / 2) * w
                y1 = (cy - bh / 2) * h
                x2 = (cx + bw / 2) * w
                y2 = (cy + bh / 2) * h
                boxes.append([x1, y1, x2, y2])
                labels.append(int(cls) + 1)

        img = TF.to_tensor(img)

        orig_h, orig_w = img.shape[1], img.shape[2]
        img = TF.resize(img, [self.imgsz, self.imgsz])

        if len(boxes) == 0:
            boxes  = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,),   dtype=torch.int64)
        else:
            boxes  = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)
            scale_x = self.imgsz / orig_w
            scale_y = self.imgsz / orig_h
            boxes[:, [0, 2]] *= scale_x
            boxes[:, [1, 3]] *= scale_y

        target = {"boxes": boxes, "labels": labels}
        return img, target


def collate_fn(batch):
    return tuple(zip(*batch))

val_dataset = PollenDatasetFromTxt(POLLEN_VAL_TXT, TARGET_CLASSES, imgsz=IMGSZ)
val_loader  = DataLoader(val_dataset, batch_size=BATCH_SIZE,
                         shuffle=False, collate_fn=collate_fn, num_workers=2)
print(f"Val images: {len(val_dataset)}")

In [2]:
# ── Evaluation ────────────────────────────────────────────────────────────────
metric = MeanAveragePrecision(iou_type="bbox", class_metrics=True)  # ← add this

with torch.no_grad():
    for images, targets in val_loader:
        images = [img.to(DEVICE) for img in images]

        predictions = model(images)

        preds_filtered = []
        for pred in predictions:
            keep = pred["scores"] >= SCORE_THRESHOLD
            preds_filtered.append({
                "boxes":  pred["boxes"][keep].cpu(),
                "scores": pred["scores"][keep].cpu(),
                "labels": pred["labels"][keep].cpu(),
            })

        targets_cpu = [{"boxes": t["boxes"].cpu(),
                        "labels": t["labels"].cpu()} for t in targets]

        metric.update(preds_filtered, targets_cpu)

results = metric.compute()
print("\n=== Evaluation Results ===")
print(f"mAP@50:    {results['map_50']:.4f}")
print(f"mAP@50-95: {results['map']:.4f}")
print(f"mAP@75:    {results['map_75']:.4f}")

NOTE! Installing ujson may make loading annotations faster.

=== Evaluation Results ===
mAP@50:    0.5499
mAP@50-95: 0.4606
mAP@75:    0.5059


In [3]:
print(results.keys())
print("map_per_class:", results["map_per_class"])
print("mar_100_per_class:", results["mar_100_per_class"])

dict_keys(['map', 'map_50', 'map_75', 'map_small', 'map_medium', 'map_large', 'mar_1', 'mar_10', 'mar_100', 'mar_small', 'mar_medium', 'mar_large', 'map_per_class', 'mar_100_per_class', 'classes'])
map_per_class: tensor([ 0.3930,  0.6353,  0.8283,  0.7623,  0.6722,  0.0000,  0.5753,  0.6771,
         0.2843,  0.3624,  0.4745,  0.6660,  0.8097,  0.7732,  0.6557,  0.7921,
         0.8730,  0.5664,  0.2241,  0.2839,  0.2094,  0.1983,  0.7897,  0.9124,
         0.5525,  0.7872,  0.7367,  0.3271,  0.0000,  0.0000,  0.4597,  0.0000,
         0.0000,  0.5929,  0.6417, -1.0000,  0.0000,  0.7086,  0.0000,  0.5618,
         0.0000,  0.7059,  0.3141,  0.0000])
mar_100_per_class: tensor([ 0.5390,  0.7000,  0.8858,  0.8497,  0.8453,  0.0000,  0.6174,  0.7366,
         0.3000,  0.3750,  0.5554,  0.6955,  0.8463,  0.8158,  0.7193,  0.8139,
         0.9075,  0.6241,  0.2250,  0.3111,  0.2929,  0.2286,  0.8333,  0.9336,
         0.7000,  0.8537,  0.7698,  0.3524,  0.0000,  0.0000,  0.7288,  0.0000,
   